# Script para preuebas de balance
Incluye código para poder justifciar lo siguiente:
1. Common support
2. SMD Love Plots

In [1]:
import os
import sys
import pickle
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Agregar el directorio base (scripts) al path para importar las clases
sys.path.append(os.path.abspath('..'))

from ps_features_builder import PSFeaturesBuilder
from ps_matching import PSMatching

In [2]:
# Definición de rutas a los archivos
PATH_DATA = '../../data/'
PATHS = {
    'vialidades': os.path.join(PATH_DATA, 'vialidades.json'),
    'speed_cameras': os.path.join(PATH_DATA, 'fotocivicas-ubicacion-puntos', 'fotocivicas-ubicacion-puntos.shp'),
    'metro_coordinates': os.path.join(PATH_DATA, 'metro-station-coordinates.parquet'),
    'afluencia_metro': os.path.join(PATH_DATA, 'afluencia-metro-semanal.parquet'),
    'classified_incidents': os.path.join(PATH_DATA, 'classified-incidents.parquet'),
    'volumen_mensual': os.path.join(PATH_DATA, 'volumen-total-mensual.parquet')
}

In [3]:
# grid_type, radio, n_circles
radios = [
    ("circular", 37.5, 25), # 0
    ("circular", 50, 15), # 1
    ("circular", 100, 20), # 2
    ("circular", 150, 15), # 3
    ("circular", 200, 20), # 4
    ("circular", 250, 25), # 5
]

## 1. Common Support
Esto luego lo paso a una tabla con los resultados para mostrar el common support y explicar cuáles son las unidades que voy a dejar fuera del análisis

In [ ]:
grid_type, radio, n_circles = radios[0]

# vemos si ya existe el archivo de PSFeaturesBuilder que estamos buscando
object_path = os.path.join(PATH_DATA, "pickle_objects", "ps_features_builder", f"{grid_type}_{int(radio)}_{n_circles}.pkl")
if os.path.exists(object_path):
    with open(object_path, "rb") as f:
        ps_builder : PSFeaturesBuilder = pickle.load(f)
else:
    print("Building new object")
    ps_builder = PSFeaturesBuilder(
        paths=PATHS,
        grid_type=grid_type,
        circle_radius=radio,
        n_circles=n_circles
    )
    ps_builder.build()
    with open(object_path, "wb") as f:
        pickle.dump(ps_builder, f)

In [ ]:
psm = PSMatching(
    ps_features=ps_builder.ps_features,
    grid=ps_builder.grid,
    outcome=ps_builder.outcome,
    grid_type='circular',
    grid_size=ps_builder.circle_radius
)
psm.build()

In [ ]:
(
    psm
    .get_common_support_table(n_bins=20)
    .pipe(lambda df: df[df.treatment > 0])
    .reset_index()
    .rename({
        "index":"PS Range",
        "treatment":"Tratamiento",
        "control":"Control"
    }, axis=1)
    .to_clipboard(index=False)
)

## 2. SMD
Aquí hago el match y las gráficas del SMD para mostra que las distribuciones de las covariables son similares tras haber hecho el match

In [4]:
path_smd_plots = "graphs/smd"

for grid_type, radio, n_circles in radios:
    # vemos si ya existe el archivo de PSFeaturesBuilder que estamos buscando
    object_path = os.path.join(PATH_DATA, "pickle_objects", "ps_features_builder", f"{grid_type}_{int(radio)}_{n_circles}.pkl")
    if os.path.exists(object_path):
        with open(object_path, "rb") as f:
            ps_builder : PSFeaturesBuilder = pickle.load(f)
    else:
        print(f"Object {int(radio)}, {n_circles} does not exist")
        continue

    psm = PSMatching(
        ps_features=ps_builder.ps_features,
        grid=ps_builder.grid,
        outcome=ps_builder.outcome,
        grid_type='circular',
        grid_size=ps_builder.circle_radius
    )
    # eliminamos las unidades de tratamiento para las que 
    # no existan al menos 3 unidades de control para elegir
    psm.build(min_controls_in_bin=3) 

    fig = psm.plot_smd_comparison(
        figsize=(8, 6), 
        threshold=0.1,
        save_path=os.path.join(PATH_DATA, path_smd_plots, f"{psm.grid_type}_{int(psm.grid_size)}.png")
    )
    plt.close()
    break